# 05 · Cross-Persona Convergence Analysis

Integrates outputs from notebooks 02–04 to produce a unified view of how much personas **converge** across three dimensions:

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from scipy import stats

from src.config import FIGURES, OUTPUTS, FONT_SCALE, SENT_COLORS, SENT_COLORS_3
from src.data_loading import load_annotations, parse_demographics, create_profiles
from src.convergence import compute_sentiment_agreement, compute_label_jaccard, merge_convergence_dimensions

MUTED = sns.color_palette("muted")
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams["figure.dpi"] = 150

FIGURES.mkdir(exist_ok=True)
OUTPUTS.mkdir(exist_ok=True)

df = load_annotations()
df = parse_demographics(df)
df = create_profiles(df)
print(f"Records: {len(df):,} | Images: {df['image_id'].nunique():,} | Personas: {df['persona_id'].nunique():,}")

## Correlation: justification cosine vs. perception Jaccard

In [ ]:
from scipy.stats import pearsonr as _pearsonr
_just_path    = OUTPUTS / "ic_profile_sim_just.npy"
_jaccard_path = OUTPUTS / "ic_profile_sim_jaccard.npy"
if _just_path.exists() and _jaccard_path.exists():
    _just_mat    = np.load(_just_path)
    _jaccard_mat = np.load(_jaccard_path)
    _n = _just_mat.shape[0]
    _idx = np.triu_indices(_n, k=1)
    _r, _p = _pearsonr(_just_mat[_idx], _jaccard_mat[_idx])
    print(f"Pearson r(justification cosine, Jaccard) = {_r:.4f}  "
          f"(p = {_p:.3e})  n = {len(_just_mat[_idx])} off-diagonal pairs")
else:
    print("Similarity cache not yet generated — run generate_figures.py first.")


## 1 · Load pre-computed similarity from notebook 03

In [ ]:
sim_df = pd.read_csv(OUTPUTS / "per_image_cosine_similarity.csv")
sim_df = sim_df.rename(columns={"mean_sim": "caption_sim"})
print(f"Image-level caption similarity: {len(sim_df):,} images")

## 2 · Compute per-image sentiment agreement

In [ ]:
sent_agree_df = compute_sentiment_agreement(df)
print(f"Sentiment agreement computed for {len(sent_agree_df):,} images")
sent_agree_df.describe().round(4)

## 3 · Compute per-image perception label agreement (Jaccard)

In [ ]:
label_agree_df = compute_label_jaccard(df)
print(f"Label Jaccard computed for {len(label_agree_df):,} images")
label_agree_df.describe().round(4)

## 4 · Merge all three dimensions

In [ ]:
conv_df = (
    sim_df[["image_id", "caption_sim", "n_personas"]]
    .merge(sent_agree_df[["image_id", "sentiment_agreement", "majority_sentiment"]], on="image_id")
    .merge(label_agree_df, on="image_id")
)
conv_df.to_csv(OUTPUTS / "convergence_all_dimensions.csv", index=False)
print(f"Convergence dataframe: {conv_df.shape}")
conv_df.describe().round(4)

## 8 · Kruskal-Wallis test across sentiment groups

In [ ]:
order = ["Positive", "Neutral", "Negative"]
sub = conv_df[conv_df["majority_sentiment"].isin(order)]

results = []
for col in ["caption_sim", "sentiment_agreement", "label_jaccard"]:
    groups = [sub[sub["majority_sentiment"] == s][col].dropna() for s in order]
    stat, p = stats.kruskal(*groups)
    results.append({"dimension": col, "H_statistic": stat, "p_value": p})
    print(f"{col}: H={stat:.4f}, p={p:.4e}")

pd.DataFrame(results).to_csv(OUTPUTS / "kruskal_wallis_convergence.csv", index=False)

## 9 · Add justification similarity as a 4th convergence dimension

In [ ]:
JUST_SIM_PATH = OUTPUTS / "per_image_just_similarity.csv"

if JUST_SIM_PATH.exists():
    just_sim_df = pd.read_csv(JUST_SIM_PATH)
    just_sim_df = just_sim_df.rename(columns={"mean_just_sim": "just_sim"})
    print(f"Loaded justification similarity: {len(just_sim_df):,} images")
    just_sim_df.describe().round(4)
else:
    print("Run notebook 03 sections 8+ first to generate per_image_just_similarity.csv")
    just_sim_df = None


In [ ]:
if just_sim_df is not None:
    conv4_df = conv_df.merge(
        just_sim_df[["image_id", "just_sim"]], on="image_id", how="left"
    )
    conv4_df.to_csv(OUTPUTS / "convergence_all_dimensions_4d.csv", index=False)
    print("4-dimension convergence dataframe:")
    print(conv4_df[["caption_sim", "sentiment_agreement", "label_jaccard", "just_sim"]].describe().round(4))
